In [3]:
import torch
import torch.nn as nn
import timm
import numpy as np

class DisjointMaskGenerator:
    def __init__(self, num_patches=225, mask_ratio=0.5):
        """
        120x120 image with 8x8 patches = 15x15 grid = 225 total patches.
        mask_ratio: Kitne % patches ko hide karna hai (e.g., 0.5 = 50%)
        """
        self.num_patches = num_patches
        self.num_mask = int(mask_ratio * num_patches)

    def generate_masks(self, batch_size):
        # Create random masks for a batch
        # 1 means Masked (Hidden), 0 means Visible
        s2_masks = np.zeros((batch_size, self.num_patches), dtype=bool)
        s1_masks = np.zeros((batch_size, self.num_patches), dtype=bool)

        for i in range(batch_size):
            # Randomly shuffle patch indices (0 to 224)
            indices = np.random.permutation(self.num_patches)
            
            # S2 (Optical) mein first 50% hide karo
            s2_hidden_idx = indices[:self.num_mask]
            s2_masks[i, s2_hidden_idx] = True
            
            # S1 (SAR) mein bache hue 50% hide karo (Disjoint masking!)
            s1_hidden_idx = indices[self.num_mask:]
            s1_masks[i, s1_hidden_idx] = True
            
        return torch.tensor(s2_masks), torch.tensor(s1_masks)

class Antariksh_ViT_Encoders(nn.Module):
    def __init__(self):
        super().__init__()
        # S1 Encoder (SAR): 2 Channels input
        # FIX: Use 'vit_tiny_patch16_224' as the base name, timm will automatically 
        # override it with our custom img_size=120 and patch_size=8
        self.s1_encoder = timm.create_model(
            'vit_tiny_patch16_224', 
            img_size=120, patch_size=8, in_chans=2, num_classes=0
        )
        
        # S2 Encoder (Optical): 10 Channels input
        self.s2_encoder = timm.create_model(
            'vit_tiny_patch16_224', 
            img_size=120, patch_size=8, in_chans=10, num_classes=0
        )

    def forward(self, s1_imgs, s2_imgs):
        # Get Latent Representations from the ViT backbones
        s1_latent = self.s1_encoder.forward_features(s1_imgs) # Shape: (Batch, 225 patches + 1 cls token, Embed_Dim)
        s2_latent = self.s2_encoder.forward_features(s2_imgs)
        return s1_latent, s2_latent
        
# --- Test the Model & Masking ---
if __name__ == "__main__":
    # Simulate the data we got from your DataLoader
    batch_size = 32
    print("Simulating Batch from DataLoader...")
    dummy_s1 = torch.randn(batch_size, 2, 120, 120)
    dummy_s2 = torch.randn(batch_size, 10, 120, 120)
    
    # 1. Test Mask Generator
    mask_gen = DisjointMaskGenerator()
    s2_mask, s1_mask = mask_gen.generate_masks(batch_size)
    print(f"\nS2 Mask Shape: {s2_mask.shape} | S1 Mask Shape: {s1_mask.shape}")
    print(f"Number of hidden patches in S2 (first image): {s2_mask[0].sum().item()} out of 225")
    
    # 2. Test Encoders
    print("\nInitializing Antariksh Encoders (ViT)...")
    model = Antariksh_ViT_Encoders()
    s1_features, s2_features = model(dummy_s1, dummy_s2)
    
    print(f"S1 Latent Features Shape: {s1_features.shape}")
    print(f"S2 Latent Features Shape: {s2_features.shape}")
    print("\nPhase 2 is Complete! ✅ Both modalities are successfully mapped into the Latent Space.")

Simulating Batch from DataLoader...

S2 Mask Shape: torch.Size([32, 225]) | S1 Mask Shape: torch.Size([32, 225])
Number of hidden patches in S2 (first image): 112 out of 225

Initializing Antariksh Encoders (ViT)...
S1 Latent Features Shape: torch.Size([32, 226, 192])
S2 Latent Features Shape: torch.Size([32, 226, 192])

Phase 2 is Complete! ✅ Both modalities are successfully mapped into the Latent Space.
